# Notebook 3 — Análise de Modelos

# Capstone Project | Liga <a class="anchor" id="Cpstone"></a>

**Authors:**\
Maria Margarida Mouteira Guerreiro Romero Antelo <a20243063@pbs.up.pt>\
Mariana Chaves Santos <a20243060@pbs.up.pt>\
Mónica Isabel da Silva Simões <a20250187@pbs.up.pt>\
Hugo Miguel Carvalho da Silva Glória <a20250093@pbs.up.pt>\
Rui Alexandre Paiva Santos <a20160686@pbs.up.pt>
>
> Publication Date: October 2025

**Table of contents:**
1. [Install required packages](#required_packages)
2. [Define program variables and setup db connection](#operators)
3. [Declare database manager helping functions](#db_manager)
4. [Declare technician assignment function](#logical_operators)
5. [Declare random intervencion close](#comparison_operators)
6. [Main program](#main)

---

Rat, Shr e Rch — são métricas fundamentais no mundo da medição de audiência televisiva. Eles ajudam a entender como um programa, canal ou campanha publicitária está performando, mas cada um foca em um aspecto diferente.

RAT (Rating)
O Rating (também conhecido como Rat%) é a métrica mais tradicional e mede a porcentagem do universo total de TVs ligadas e desligadas que está sintonizada em um determinado programa ou canal.

O que ele mostra: A popularidade absoluta de um programa. Se um programa tem um Rating de 5%, significa que 5% de todas as TVs na área de pesquisa (incluindo as desligadas) estão assistindo a esse programa.

Como é calculado: Número de pessoas ou domicílios assistindo a um programa / Universo total de pessoas ou domicílios.

Para que serve: É usado para comparar o desempenho de programas em diferentes horários e para negociar o valor de espaços publicitários, já que indica o tamanho da audiência potencial.

SHR (Share)
O Share (ou Shr%) mede a porcentagem de audiência de um programa ou canal em relação apenas às TVs que estão ligadas no momento.

O que ele mostra: A "fatia de mercado" que um programa ou canal tem. Se um programa tem um Share de 20%, significa que 20% das pessoas que estão assistindo TV naquele momento estão sintonizadas nele.

Como é calculado: Audiência do programa / Total de TVs ligadas.

Para que serve: É útil para entender o desempenho de um programa em relação à concorrência direta no mesmo horário. Um Share alto indica que o programa é líder de audiência, mesmo que o número total de pessoas assistindo TV naquele momento seja menor.

RCH (Reach)
O Reach (ou Rch%) mede o alcance de uma campanha publicitária ou programa, mostrando a porcentagem de indivíduos que foram expostos ao conteúdo por pelo menos um minuto, sem contar a duplicação.

O que ele mostra: O tamanho da audiência única que foi alcançada. Se uma campanha publicitária teve um Reach de 40%, significa que 40% das pessoas no universo pesquisado viram pelo menos um dos anúncios.

Como é calculado: Número de pessoas únicas que assistiram ao conteúdo / Universo total de pessoas.

Para que serve: É crucial para planejar campanhas publicitárias, pois indica a capacidade de um programa ou canal de atingir um grande número de pessoas diferentes.

Exemplo Prático
Imagine uma cidade com 1.000 TVs. Em um determinado momento:

500 TVs estão ligadas.

200 TVs estão sintonizadas no Canal A.

Nesse caso:

RAT: O Canal A tem um Rating de 20% (200 TVs / 1.000 TVs). Isso mostra que 20% de todas as TVs da cidade estão assistindo a ele.

SHR: O Canal A tem um Share de 40% (200 TVs / 500 TVs). Isso mostra que ele tem 40% da audiência entre as pessoas que estão com a TV ligada.

Agora, se a campanha publicitária do Canal A teve um RCH de 50%, significa que, ao longo de sua duração, ela alcançou 500 pessoas únicas (50% de 1.000), mesmo que algumas delas tenham assistido a mais de um anúncio.

## Setup working directories <a class="anchor" id="folder_structure"></a>
---

In [11]:
import sys 
import os 
from dotenv import load_dotenv

# Determinar o diretório raiz do projeto
current_dir = os.getcwd()  
project_root = os.path.dirname(current_dir)
    
# Define paths baseados na estrutura do container
PythonScriptsPath = os.path.join(project_root, 'Python')
DataFilesPath = os.path.join(project_root, 'Base de Dados')
NotebooksPath = os.path.join(project_root, 'notebooks')
log_path = os.path.join(project_root, 'logs')

# Criar diretório de logs se não existir
os.makedirs(log_path, exist_ok=True)

# Carregar .env se existir
dotenv_path = os.path.join(project_root, '.env')
if os.path.exists(dotenv_path):
    load_dotenv(dotenv_path)
    print("✓ Loaded environment variables from .env file.")

# Adicionar paths ao sys.path
paths_to_add = [
    ('PythonScriptsPath', PythonScriptsPath),
    ('project_root', project_root),
    ('NotebooksPath', NotebooksPath)
]

# DataFilesPath normalmente não precisa estar no sys.path (é só para dados)
# Mas se tiver scripts Python lá, pode adicionar

for name, path in paths_to_add:
    if os.path.exists(path):
        if path not in sys.path:
            sys.path.insert(0, path)
            print(f"✓ Added '{name}' to path: {path}")
    else:
        print(f"⚠ '{name}' não existe: {path}")


# Crie o ficheiro __init__.py no seu notebook
init_file_path = os.path.join(PythonScriptsPath, '__init__.py')

if not os.path.exists(init_file_path):
    with open(init_file_path, 'w') as f:
        pass  # Cria um ficheiro vazio
    print(f"✓ Ficheiro __init__.py criado em: {init_file_path}")


# Verificação de configuração
print(f"\n{'='*60}")
print("CONFIGURAÇÃO DE PATHS")
print(f"{'='*60}")
print(f"Working directory:   {os.getcwd()}")
print(f"Project root:        {project_root}")
print(f"Notebooks:           {NotebooksPath}")
print(f"Python scripts:      {PythonScriptsPath}")
print(f"Data files:          {DataFilesPath}")
print(f"Logs:                {log_path}")
print(f"{'='*60}\n")



CONFIGURAÇÃO DE PATHS
Working directory:   /home/jovyan/notebooks
Project root:        /home/jovyan
Notebooks:           /home/jovyan/notebooks
Python scripts:      /home/jovyan/Python
Data files:          /home/jovyan/Base de Dados
Logs:                /home/jovyan/logs




## Import required modules <a class="anchor" id="required_modules"></a>
---

In [12]:
# Personalized imports
from HelperFunctions import ( # type: ignore
    Logger as log,
    DataCleaning as dc,
    FileOperations as fo,
    DataValidation as dv,
    DataAnalysis as da,
    Distance as dist,
    DatabaseManager as dbm,
    LigaSchedule as ls
)


# Standard library imports
import getpass
import importlib.util
import re
from collections import defaultdict, Counter
from datetime import timedelta, datetime, time
from itertools import combinations

# Third-party imports
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from geopy.distance import geodesic
from IPython.display import display
from joblib import dump, load
import json
from lightgbm import LGBMRegressor
from scipy import stats
from statsmodels.stats.multitest import multipletests
from typing import Optional, List, Dict, Any
from xgboost import XGBRegressor
import mlflow
import mlflow.genai
from mlflow.tracking import MlflowClient
from mlflow.entities import Prompt
import requests



# Scikit-learn imports
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

os.environ['MLFLOW_TRACKING_URI'] = 'http://mlflow:5000'
mlflow.set_tracking_uri('http://mlflow:5000')

print(f"🎯 MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print("🚀 Ready to track experiments!")

AttributeError: partially initialized module 'mlflow' from '/opt/conda/lib/python3.13/site-packages/mlflow/__init__.py' has no attribute 'version' (most likely due to a circular import)

In [ ]:
# we'll disable the trace UI 
mlflow.tracing.disable_notebook_display()

# Display MLflow version to confirm we're using 3.4.0
print(f"🔍 MLflow Version: {mlflow.__version__}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

🔍 MLflow Version: 3.4.0
📅 Date: 2025-10-07 22:37:40



## Main program <a class="anchor" id="main"></a>
---

In [ ]:
# Cria o log do programa principal
logger_main = log.setup_logger('main')

# Carregamento da tabela via csv, com todos os valores iniciais da liga

# Define the path to the stadia file
base_data_file_path = DataFilesPath + '/liga_base_data.csv'


logger_main.info("Carregar os dados da liga...")

# Bloco de tratamento de erros para carregamento do ficheiro
try:
    logger_main.info(f"Tentando carregar ficheiro de dados da liga: {base_data_file_path}")

    # Tentar carregar o ficheiro CSV
    logger_main.info("Carregando dados da liga com separador ';' e decimal '.'...")
    df_processed = fo.safe_read_csv(base_data_file_path, logger_main, sep=';', decimal='.', low_memory=False)

    if df_processed.empty:
        logger_main.warning("O ficheiro de dados da liga foi carregado mas está vazio.")
        raise ValueError("O ficheiro de dados da liga está vazio.")

    # Verificar se as colunas necessárias existem
    required_columns = ['Competicao', 'Lotacao_Casa', 'Latitude_Casa', 'Longitude_Casa']
    missing_columns = [col for col in required_columns if col not in df_processed.columns]

    if missing_columns:
        logger_main.error(f"Colunas obrigatórias em falta: {missing_columns}")
        raise ValueError(f"O ficheiro de dados da liga não contém as colunas necessárias: {missing_columns}")

    logger_main.info("Todas as colunas necessárias estão presentes no ficheiro.")

    #print("\nDataFrame with location information:")
    #print(df_processed)
    print(f"\nNúmero de linhas no df_processed: {len(df_processed)}")

except pd.errors.ParserError as e:
    logger_main.error(f"Erro ao analisar o ficheiro CSV: {e}")
    raise
except Exception as e:
    logger_main.error(f"Erro desconhecido ao carregar o ficheiro: {e}")
    raise

INFO - O log 'logs/main.log' é recente. O novo conteúdo será anexado.

Número de linhas no df_processed: 9614


In [ ]:
# Configuração do MLflow
EXPERIMENT_NAME = "Liga Portuguesa - Previsão de Assistência"

def get_tracking_uri():
    """Tenta automaticamente descobrir o servidor MLflow"""
    uris_to_try = [
        "http://mlflow:5000",      # rede interna Docker (containers)
        "http://localhost:5001",   # host externo (porta mapeada)
        "http://127.0.0.1:5001"    # fallback para host externo
    ]

    for uri in uris_to_try:
        try:
            # tenta o endpoint /health primeiro (mais leve que /experiments)
            r = requests.get(f"{uri}/health", timeout=3)
            if r.status_code == 200:
                print(f"✅ Ligado a MLflow em {uri}")
                return uri
        except Exception as e:
            print(f"⚠️  Não consegui ligar a {uri}: {e}")

    raise RuntimeError("❌ Não consegui encontrar nenhum servidor MLflow ativo.")

# Descobrir automaticamente o servidor
MLFLOW_TRACKING_URI = get_tracking_uri()
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Criar ou obter experimento
try:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
        print(f"📝 Criado novo experimento: {EXPERIMENT_NAME}")
    else:
        experiment_id = experiment.experiment_id
        print(f"📂 Usar experimento existente: {EXPERIMENT_NAME}")
except Exception as e:
    print(f"⚠️  A usar experimento por defeito devido a: {e}")
    experiment_id = "0"

# Definir experimento
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"🎯 Experiment ID: {experiment_id}")
print(f"🔗 MLflow UI: {MLFLOW_TRACKING_URI}")


✅ Ligado a MLflow em http://mlflow:5000
📂 Usar experimento existente: Liga Portuguesa - Previsão de Assistência
🎯 Experiment ID: 1
🔗 MLflow UI: http://mlflow:5000


# **# New time Series Split --> Custom Football Time Split ⚽**

**MARIANA** - comentei este porque dava erro (derivado a ter variáveis em formato datetime) é necessário confirmar se é para manter o que está a seguir (funcional)

In [ ]:
'''

# =============================================================================
# CUSTOM FOOTBALL SEASON SPLIT CLASS
# =============================================================================

class FootballSeasonSplit:
    """
    Custom cross-validation splitter para dados de futebol por época.

    Treina com todas as épocas até uma determinada época e testa na época seguinte.
    Implementa expanding window: cada fold adiciona mais uma época aos dados de treino.
    """

    def __init__(self, start_test_season=201819):
        """
        Args:
            start_test_season (int): Primeira época a ser usada para teste (formato: 201819)
        """
        self.start_test_season = start_test_season

    def split(self, X, y=None, groups=None, season_column='Epoca'):
        """
        Gera os índices de treino e teste para cada fold.

        Args:
            X (pd.DataFrame): Features com coluna de época
            y: Target (não usado, mantido para compatibilidade sklearn)
            groups: Groups (não usado)
            season_column (str): Nome da coluna com a época

        Yields:
            tuple: (train_indices, test_indices)
        """
        # Se X for DataFrame, usar diretamente. Se não, assumir que época está em df_processed
        if hasattr(X, 'columns') and season_column in X.columns:
            epochs = sorted(X[season_column].unique())
        else:
            # Fallback: usar df_processed global (assumindo que existe)
            epochs = sorted(df_processed[season_column].unique())

        # GARANTIR que épocas são inteiros
        epochs = [int(epoch) for epoch in epochs]

        # Encontra épocas disponíveis para teste (a partir de start_test_season)
        test_epochs = [epoch for epoch in epochs if epoch >= self.start_test_season]

        if not test_epochs:
            raise ValueError(f"Nenhuma época encontrada >= {self.start_test_season}")

        print(f"\n🏆 ESTRATÉGIA IMPLEMENTADA:")
        print(f"   📚 Treino inicial: Todas as épocas até 2017/18")
        print(f"   🎯 Épocas para teste: {test_epochs}")
        print(f"   📊 Total de folds: {len(test_epochs)}")
        print("=" * 70)

        for test_epoch in test_epochs:
            # Épocas de treino: todas as épocas anteriores à época de teste
            train_epochs = [epoch for epoch in epochs if epoch < test_epoch]

            if not train_epochs:
                print(f"⚠️  Aviso: Nenhuma época de treino disponível para testar {test_epoch}")
                continue

            # Usar df_processed para obter índices corretos
            train_idx = df_processed[df_processed[season_column].isin(train_epochs)].index.values
            test_idx = df_processed[df_processed[season_column] == test_epoch].index.values

            print(f"📈 FOLD {len(test_epochs) - len([e for e in test_epochs if e >= test_epoch]) + 1}: "
                  f"Train(até {max(train_epochs)}) → Test({test_epoch})")
            print(f"   • Épocas de treino: {train_epochs}")
            print(f"   • Amostras treino: {len(train_idx)}, Amostras teste: {len(test_idx)}")

            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None, season_column='Epoca'):
        """Retorna o número de folds."""
        if hasattr(X, 'columns') and season_column in X.columns:
            epochs = sorted(X[season_column].unique())
        else:
            epochs = sorted(df_processed[season_column].unique())

        epochs = [int(epoch) for epoch in epochs]
        test_epochs = [epoch for epoch in epochs if epoch >= self.start_test_season]
        return len(test_epochs)


def plot_football_season_splits(X, splitter, season_column='Epoca'):
    """
    Visualiza os splits por época de futebol - versão simplificada.
    """
    epochs = sorted(df_processed[season_column].unique())
    n_splits = splitter.get_n_splits(X, season_column=season_column)

    if n_splits == 0:
        print("⚠️  Nenhum split disponível para visualizar")
        return

    fig, axs = plt.subplots(n_splits, 1, figsize=(15, 3 * n_splits), sharex=True)
    if n_splits == 1:
        axs = [axs]

    fold_num = 0
    for train_idx, test_idx in splitter.split(X, season_column=season_column):
        ax = axs[fold_num]

        # Informações do fold atual
        train_seasons = sorted(df_processed.iloc[train_idx][season_column].unique())
        test_season = df_processed.iloc[test_idx][season_column].iloc[0]

        # Criar visualização simples
        all_seasons = sorted(epochs)
        x_pos = list(range(len(all_seasons)))

        # Plot todas as épocas como background
        ax.bar(x_pos, [1]*len(all_seasons), color='lightgray', alpha=0.3, label='Não usado')

        # Plot épocas de treino
        train_positions = [i for i, season in enumerate(all_seasons) if season in train_seasons]
        ax.bar(train_positions, [1]*len(train_positions), color='blue', alpha=0.7, label='Treino')

        # Plot época de teste
        test_position = [i for i, season in enumerate(all_seasons) if season == test_season]
        ax.bar(test_position, [1]*len(test_position), color='red', alpha=0.8, label='Teste')

        # Configurações
        ax.set_title(f'Fold {fold_num + 1}: Train({min(train_seasons)}-{max(train_seasons)}) → Test({test_season})')
        ax.set_xticks(x_pos)
        ax.set_xticklabels([str(s) for s in all_seasons], rotation=45)
        ax.set_ylabel('Uso dos Dados')
        ax.set_ylim(0, 1.2)

        # Adicionar informações
        ax.text(0.02, 0.95, f'Treino: {len(train_idx)} jogos\nTeste: {len(test_idx)} jogos',
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        if fold_num == 0:  # Legend apenas no primeiro
            ax.legend(loc='upper right')

        fold_num += 1

    plt.xlabel('Épocas')
    plt.suptitle('🏆 Custom Football Season Split - Expanding Window', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


# =============================================================================
# --- 4. Model Selection and Data Splitting (VERSÃO ATUALIZADA) ---
# =============================================================================

print("\n--- 4. Model Selection and Data Splitting (Football Season Split) ---")

# Create Macro Regions before dropping columns
# Criar macro regiões para efeitos analíticos
def categorizar_macro_regiao(regiao_simples):
    if regiao_simples in ['Norte']:
        return 'Norte'
    elif regiao_simples in ['Centro', 'Lisboa e Vale do Tejo']:
        return 'Centro'
    elif regiao_simples in ['Alentejo', 'Algarve']:
        return 'Sul'
    else:
        return 'Outros'

# Aplicar macro-regiões
df_processed['Macro_Regiao_Casa'] = df_processed['Regiao_Simples_Casa'].apply(categorizar_macro_regiao)
df_processed['Macro_Regiao_Fora'] = df_processed['Regiao_Simples_Fora'].apply(categorizar_macro_regiao)

# Drop columns (IMPORTANTE: Manter 'Epoca' para o split personalizado)
columns_to_drop = [
    'Latitude_Casa',
    'Latitude_Fora',
    'Longitude_Fora',
    'Longitude_Casa',
    'Regiao_Simples_Casa',
    'Regiao_Simples_Fora',
    'Distancia_km_bin',
    'Hour'  # Dropping Hour as 'Part_of_day' is a categorical representation
    # NÃO remover 'Epoca' ainda - precisamos dela para o split
]
df_processed_model = df_processed.drop(columns=columns_to_drop)
print(f"✅ Dropped columns: {columns_to_drop}")
print(f"✅ Mantida coluna 'Epoca' para split personalizado")

# Identify categorical columns for one-hot encoding
categorical_cols = df_processed_model.select_dtypes(include=['object', 'category']).columns.tolist()
target_columns = ['Espetadores']
categorical_cols = [col for col in categorical_cols if col not in target_columns + ['Epoca']]  # Excluir 'Epoca' do encoding

print(f"📊 Categorical columns for one-hot encoding: {categorical_cols}")

# Apply one-hot encoding to categorical columns
df_processed_model = pd.get_dummies(df_processed_model, columns=categorical_cols, dummy_na=False)
print("✅ One-hot encoding applied to categorical columns.")

# IMPORTANTE: Converter 'Epoca' para inteiro se necessário
print(f"\n📅 Tipo da coluna 'Epoca': {df_processed_model['Epoca'].dtype}")
print(f"📅 Épocas disponíveis (antes conversão): {sorted(df_processed_model['Epoca'].unique())}")

# Converter para inteiro se estiver como string
if df_processed_model['Epoca'].dtype == 'object' or df_processed_model['Epoca'].dtype == 'string':
    print("🔄 Convertendo coluna 'Epoca' de string para inteiro...")
    df_processed_model['Epoca'] = df_processed_model['Epoca'].astype(int)
    df_processed['Epoca'] = df_processed['Epoca'].astype(int)  # Também no DataFrame original
    print("✅ Conversão concluída!")

print(f"📅 Épocas disponíveis (após conversão): {sorted(df_processed_model['Epoca'].unique())}")
print(f"📊 Distribuição por época:")
print(df_processed_model['Epoca'].value_counts().sort_index())

# Separate features (X) and target (y)
# IMPORTANTE: Manter 'Epoca' no X para o split funcionar
X = df_processed_model.drop(columns=target_columns + ['Data_Hora'])  # Manter 'Epoca'
y = df_processed_model[target_columns]

print(f"✅ Features (X): {X.shape} - Inclui coluna 'Epoca'")
print(f"✅ Target (y): {y.shape}")

# =============================================================================
# IMPLEMENTAR O SPLIT PERSONALIZADO
# =============================================================================

print("\n🏆 === CONFIGURANDO CUSTOM FOOTBALL SEASON SPLIT ===")

# Criar o splitter personalizado
# ESTRATÉGIA: Treinar com tudo até 2017/2018, testar a partir de 2018/2019
available_epochs = sorted(df_processed_model['Epoca'].unique())
print(f"Épocas disponíveis: {available_epochs}")

# FIXO: Primeira época de teste deve ser 2018/2019 (201819)
start_test_season = 201819

# Verificar se temos dados suficientes para esta estratégia
train_epochs_available = [epoch for epoch in available_epochs if epoch <= 201718]
test_epochs_available = [epoch for epoch in available_epochs if epoch >= 201819]

print(f"📚 Épocas para treino inicial (≤2017/18): {train_epochs_available}")
print(f"🎯 Épocas para teste (≥2018/19): {test_epochs_available}")

if not train_epochs_available:
    print("❌ ERRO: Não há dados de épocas ≤ 2017/18 para treino inicial!")
    print("   Ajuste start_test_season ou verifique seus dados.")
    # Fallback para primeira época disponível + 1
    if len(available_epochs) > 1:
        start_test_season = available_epochs[1]
        print(f"🔄 Usando fallback: primeira época de teste = {start_test_season}")
    else:
        raise ValueError("Dados insuficientes para split temporal")

if not test_epochs_available:
    print("❌ ERRO: Não há dados de épocas ≥ 2018/19 para teste!")

football_splitter = FootballSeasonSplit(start_test_season=start_test_season)
n_folds = football_splitter.get_n_splits(X)
print(f"✅ Custom Football Season Split configurado com {n_folds} folds")

# --- Visualization of Custom Football Season Splits ---
print("\n📊 === VISUALIZANDO OS SPLITS ===")
plot_football_season_splits(X, football_splitter)

# =============================================================================
# GRID SEARCH COM O NOVO SPLIT
# =============================================================================

print("\n🔍 === GRID SEARCH COM FOOTBALL SEASON SPLIT ===")

# Grid Search para Decision Tree com split personalizado
dt_model = DecisionTreeRegressor(random_state=42)
param_grid = {
    'max_depth': [None, 10, 15, 20, 25, 30],
    'min_samples_split': [10, 15, 20, 25, 30],
    'min_samples_leaf': [1, 2, 3, 4]
}

# Remover 'Epoca' das features para treinar os modelos
X_for_models = X.drop(columns=['Epoca'])
print(f"✅ Features para modelos (sem 'Epoca'): {X_for_models.shape}")

# IMPORTANTE: Criar um wrapper para o GridSearchCV funcionar corretamente
class FootballSeasonSplitForGridSearch:
    def __init__(self, football_splitter, X_with_epoca):
        self.football_splitter = football_splitter
        self.X_with_epoca = X_with_epoca

    def split(self, X, y=None, groups=None):
        """Wrapper para funcionar com GridSearchCV"""
        for train_idx, test_idx in self.football_splitter.split(self.X_with_epoca):
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.football_splitter.get_n_splits(self.X_with_epoca)

# Criar wrapper para GridSearch
cv_wrapper = FootballSeasonSplitForGridSearch(football_splitter, X)

# Grid Search com split personalizado
print("🚀 Iniciando Grid Search com Football Season Split...")
grid_search_dt = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid,
    cv=cv_wrapper,  # Usar o wrapper personalizado
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Fit Grid Search
grid_search_dt.fit(X_for_models, y.values.ravel())

best_dt_model = grid_search_dt.best_estimator_
print(f"✅ Best parameters for Decision Tree: {grid_search_dt.best_params_}")
print(f"✅ Best average (negative) RMSE for Decision Tree: {grid_search_dt.best_score_:.2f}")

# =============================================================================
# DEFINIR MODELOS (incluindo DT otimizada)
# =============================================================================

# Removed redundant models definition from here
# models = { ... }


# =============================================================================
# AVALIAR TODOS OS MODELOS COM O SPLIT PERSONALIZADO
# =============================================================================

print("\n🏆 === AVALIANDO TODOS OS MODELOS ===")

def evaluate_models_football_split(X, y, models, splitter):
    """Avalia modelos com o split personalizado simplificado"""
    results = []

    fold_num = 0
    for train_idx, test_idx in splitter.split(X):
        # Features sem 'Epoca'
        X_features = X.drop(columns=['Epoca'])

        X_train, X_test = X_features.iloc[train_idx], X_features.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Informações do fold
        test_season = X.iloc[test_idx]['Epoca'].iloc[0]
        train_seasons = sorted(X.iloc[train_idx]['Epoca'].unique())

        print(f"\n=== Fold {fold_num + 1}: Testando época {test_season} ===")
        print(f"Épocas de treino: {min(train_seasons)}-{max(train_seasons)}")

        for model_name, model in models.items():
            try:
                # Treinar modelo
                model.fit(X_train, y_train.values.ravel())

                # Predições
                y_pred = model.predict(X_test)

                # Métricas
                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                mae = mean_absolute_error(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)

                results.append({
                    'Fold': fold_num + 1,
                    'Test_Season': test_season,
                    'Train_Seasons': f"{min(train_seasons)}-{max(train_seasons)}",
                    'Model': model_name,
                    'MSE': mse,
                    'RMSE': rmse,
                    'MAE': mae,
                    'R2': r2,
                    'N_Train': len(X_train),
                    'N_Test': len(X_test)
                })

                print(f"  {model_name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.3f}")

            except Exception as e:
                print(f"  ❌ Erro com {model_name}: {str(e)}")

        fold_num += 1

    return pd.DataFrame(results)

# Executar avaliação
results_df = evaluate_models_football_split(X, y, models, football_splitter)

# =============================================================================
# RESUMO DOS RESULTADOS
# =============================================================================

print("\n📊 === RESUMO FINAL DOS RESULTADOS ===")

def summarize_results(results_df):
    """Cria resumo estatístico dos resultados por modelo."""
    summary = results_df.groupby('Model').agg({
        'RMSE': ['mean', 'std', 'min', 'max'],
        'MAE': ['mean', 'std', 'min', 'max'],
        'R2': ['mean', 'std', 'min', 'max']
    }).round(3)

    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    return summary.sort_values('RMSE_mean')

# Mostrar resumo
if not results_df.empty:
    summary = summarize_results(results_df)
    print("\n🏆 RANKING DOS MODELOS (por RMSE médio):")
    print("="*80)
    print(summary)

    # Mostrar resultados detalhados
    print(f"\n📋 RESULTADOS DETALHADOS POR ÉPOCA:")
    print("="*80)
    for season in sorted(results_df['Test_Season'].unique()):
        season_results = results_df[results_df['Test_Season'] == season]
        best_model = season_results.loc[season_results['RMSE'].idxmin()]
        print(f"Época {season}: Melhor modelo = {best_model['Model']} (RMSE: {best_model['RMSE']:.2f})")

print("\n🎯 *** CUSTOM FOOTBALL SEASON SPLIT IMPLEMENTADO COM SUCESSO! ***")
print("✅ Split por épocas implementado")
print("✅ Grid Search otimizado com split personalizado")
print("✅ Todos os modelos avaliados com expanding window")
print("✅ Resultados organizados por época de teste")'''

'\n\n# =============================================================================\n# CUSTOM FOOTBALL SEASON SPLIT CLASS\n# =============================================================================\n\nclass FootballSeasonSplit:\n    """\n    Custom cross-validation splitter para dados de futebol por época.\n\n    Treina com todas as épocas até uma determinada época e testa na época seguinte.\n    Implementa expanding window: cada fold adiciona mais uma época aos dados de treino.\n    """\n\n    def __init__(self, start_test_season=201819):\n        """\n        Args:\n            start_test_season (int): Primeira época a ser usada para teste (formato: 201819)\n        """\n        self.start_test_season = start_test_season\n\n    def split(self, X, y=None, groups=None, season_column=\'Epoca\'):\n        """\n        Gera os índices de treino e teste para cada fold.\n\n        Args:\n            X (pd.DataFrame): Features com coluna de época\n            y: Target (não usado, mant

# **# As correções abaixo evitam os erros com o XGBOOST E LIGHTGBM e, tentam, anular o overfitting.**

In [ ]:
# =============================================================================
# CÓDIGO INTEGRADO COM SOLUÇÃO COMPLETA DE OVERFITTING
# =============================================================================


# =============================================================================
# CLASSES E FUNÇÕES AUXILIARES
# =============================================================================

class FootballSeasonSplit:
    """Custom cross-validation splitter para dados de futebol por época."""

    def __init__(self, start_test_season=201819):
        self.start_test_season = start_test_season

    def split(self, X, y=None, groups=None, season_column='Epoca'):
        if hasattr(X, 'columns') and season_column in X.columns:
            epochs = sorted(X[season_column].unique())
        else:
            epochs = sorted(df_processed[season_column].unique())

        epochs = [int(epoch) for epoch in epochs]
        test_epochs = [epoch for epoch in epochs if epoch >= self.start_test_season]

        if not test_epochs:
            raise ValueError(f"Nenhuma época encontrada >= {self.start_test_season}")

        print(f"\n🏆 ESTRATÉGIA IMPLEMENTADA:")
        print(f"   📚 Treino inicial: Todas as épocas até 2017/18")
        print(f"   🎯 Épocas para teste: {test_epochs}")
        print(f"   📊 Total de folds: {len(test_epochs)}")
        print("=" * 70)

        for test_epoch in test_epochs:
            train_epochs = [epoch for epoch in epochs if epoch < test_epoch]

            if not train_epochs:
                print(f"⚠️  Aviso: Nenhuma época de treino disponível para testar {test_epoch}")
                continue

            train_idx = df_processed[df_processed[season_column].isin(train_epochs)].index.values
            test_idx = df_processed[df_processed[season_column] == test_epoch].index.values

            print(f"📈 FOLD {len(test_epochs) - len([e for e in test_epochs if e >= test_epoch]) + 1}: "
                  f"Train(até {max(train_epochs)}) → Test({test_epoch})")
            print(f"   • Épocas de treino: {train_epochs}")
            print(f"   • Amostras treino: {len(train_idx)}, Amostras teste: {len(test_idx)}")

            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None, season_column='Epoca'):
        if hasattr(X, 'columns') and season_column in X.columns:
            epochs = sorted(X[season_column].unique())
        else:
            epochs = sorted(df_processed[season_column].unique())

        epochs = [int(epoch) for epoch in epochs]
        test_epochs = [epoch for epoch in epochs if epoch >= self.start_test_season]
        return len(test_epochs)


def clean_column_names(df):
    """Limpa nomes de colunas para compatibilidade com XGBoost/LightGBM"""
    df_clean = df.copy()
    new_columns = []
    for col in df_clean.columns:
        new_col = re.sub(r'[<>\[\],\s]+', '_', str(col))
        new_col = re.sub(r'[^\w]', '_', new_col)
        new_col = re.sub(r'_+', '_', new_col)
        new_col = new_col.strip('_')
        new_columns.append(new_col)
    df_clean.columns = new_columns
    return df_clean


def analyze_feature_correlation(X, threshold=0.9):
    """Identifica features altamente correlacionadas"""
    print("🔍 === ANÁLISE DE CORRELAÇÃO ENTRE FEATURES ===")

    X_numeric = X.select_dtypes(include=[np.number])
    corr_matrix = X_numeric.corr().abs()

    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if corr_matrix.iloc[i, j] >= threshold:
                high_corr_pairs.append({
                    'Feature_1': corr_matrix.columns[i],
                    'Feature_2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })

    features_to_remove = set()
    if high_corr_pairs:
        print(f"⚠️  Encontradas {len(high_corr_pairs)} pares de features com correlação ≥ {threshold}")
        corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
        print(corr_df.head(10).to_string(index=False))

        for pair in high_corr_pairs[:10]:
            features_to_remove.add(pair['Feature_2'])

        print(f"\n💡 Sugestão: Remover {len(features_to_remove)} features correlacionadas")
        return corr_df, list(features_to_remove)
    else:
        print(f"✅ Nenhuma correlação alta encontrada")
        return pd.DataFrame(), []


def feature_importance_analysis(X, y):
    """Análise de importância com Random Forest"""
    print(f"\n🌳 === ANÁLISE DE IMPORTÂNCIA (RANDOM FOREST) ===")

    X_numeric = X.select_dtypes(include=[np.number])
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_numeric, y.values.ravel())

    importance_df = pd.DataFrame({
        'Feature': X_numeric.columns,
        'Importance': rf.feature_importances_
    }).sort_values('Importance', ascending=False)

    importance_df['Cumulative_Importance'] = importance_df['Importance'].cumsum()

    print("🏆 Top 15 features mais importantes:")
    print(importance_df.head(15).to_string(index=False))

    features_80pct = len(importance_df[importance_df['Cumulative_Importance'] <= 0.8])
    print(f"\n💡 {features_80pct} features explicam 80% da importância total")

    return importance_df


def feature_selection_statistical(X, y, k=50):
    """Seleção baseada em testes estatísticos"""
    print(f"\n📊 === SELEÇÃO ESTATÍSTICA (Top {k} features) ===")

    X_numeric = X.select_dtypes(include=[np.number])
    selector = SelectKBest(score_func=f_regression, k=min(k, X_numeric.shape[1]))
    X_selected = selector.fit_transform(X_numeric, y.values.ravel())

    selected_features = X_numeric.columns[selector.get_support()].tolist()

    scores = pd.DataFrame({
        'Feature': X_numeric.columns,
        'Score': selector.scores_
    }).sort_values('Score', ascending=False)

    print(f"✅ {len(selected_features)} features selecionadas")
    print("\n🏆 Top 10 features por score estatístico:")
    print(scores.head(10).to_string(index=False))

    return selected_features, scores


def comprehensive_feature_reduction(X, y, final_features_target=40):
    """Estratégia combinada para redução de features"""
    print(f"\n🎯 === ESTRATÉGIA COMBINADA DE REDUÇÃO ===")
    print(f"🎯 Meta: Reduzir para ~{final_features_target} features")
    print(f"📊 Features originais: {X.shape[1]}")

    # 1. Remover features correlacionadas
    corr_df, corr_to_remove = analyze_feature_correlation(X, threshold=0.9)

    # 2. Análise de importância
    importance_df = feature_importance_analysis(X, y)

    # 3. Seleção estatística
    selected_statistical, scores_df = feature_selection_statistical(X, y, k=final_features_target)

    # 4. Combinar critérios
    print("\n--- Combinação de Critérios ---")

    # Features importantes (top 80% da importância)
    important_features = importance_df[importance_df['Cumulative_Importance'] <= 0.8]['Feature'].tolist()

    # Intersecção entre importância e seleção estatística
    final_features = list(set(important_features) & set(selected_statistical))

    # Se muito poucos, adicionar top features por importância
    if len(final_features) < final_features_target * 0.8:
        additional = importance_df['Feature'].tolist()[:final_features_target]
        final_features = list(set(final_features + additional))

    # Remover features correlacionadas
    final_features = [f for f in final_features if f not in corr_to_remove]

    #remover features que causam leakage

    features_to_remove_leakage = [
    'Espetadores_log',           # Transformação do target
    'TaxaOcupacao_decimal',      # Calculada usando o target
]

    # Limitar ao target
    final_features = final_features[:final_features_target]

    print(f"\n✅ RESULTADO FINAL:")
    print(f"   📊 Features selecionadas: {len(final_features)}")
    print(f"   🎯 Redução: {X.shape[1] - len(final_features)} features ({((X.shape[1] - len(final_features))/X.shape[1]*100):.1f}%)")

    final_importance = importance_df[importance_df['Feature'].isin(final_features)].head(10)
    print(f"\n🏆 Top 10 features selecionadas:")
    print(final_importance[['Feature', 'Importance']].to_string(index=False))

    return final_features, {
        'correlation_analysis': corr_df,
        'importance_analysis': importance_df,
        'statistical_scores': scores_df,
        'removed_correlated': corr_to_remove
    }


# =============================================================================
# --- PREPROCESSING (IGUAL AO ORIGINAL) ---
# =============================================================================

print("\n--- 4. Model Selection and Data Splitting (Anti-Overfitting Version) ---")

# Aplicar macro-regiões
#df_processed['Macro_Regiao_Casa'] = df_processed['Regiao_Simples_Casa'].apply(categorizar_macro_regiao)
#df_processed['Macro_Regiao_Fora'] = df_processed['Regiao_Simples_Fora'].apply(categorizar_macro_regiao)

# Drop columns (IMPORTANTE: Manter 'Epoca' para o split)
columns_to_drop = [
    'Latitude_Casa', 'Latitude_Fora', 'Longitude_Fora', 'Longitude_Casa',
    'Distancia_km_bin', 'Hour'
]
df_processed_model = df_processed.drop(columns=columns_to_drop)
print(f"✅ Dropped columns: {columns_to_drop}")

# One-hot encoding
categorical_cols = df_processed_model.select_dtypes(include=['object', 'category']).columns.tolist()
target_columns = ['Espetadores']
categorical_cols = [col for col in categorical_cols if col not in target_columns + ['Epoca']]

print(f"📊 Categorical columns for one-hot encoding: {categorical_cols}")
df_processed_model = pd.get_dummies(df_processed_model, columns=categorical_cols, dummy_na=False)
print("✅ One-hot encoding applied.")

# Converter época para inteiro
print(f"\n📅 Tipo da coluna 'Epoca': {df_processed_model['Epoca'].dtype}")
if df_processed_model['Epoca'].dtype == 'object' or df_processed_model['Epoca'].dtype == 'string':
    print("🔄 Convertendo coluna 'Epoca' para inteiro...")
    df_processed_model['Epoca'] = df_processed_model['Epoca'].astype(int)
    df_processed['Epoca'] = df_processed['Epoca'].astype(int)
    print("✅ Conversão concluída!")

print(f"📅 Épocas disponíveis: {sorted(df_processed_model['Epoca'].unique())}")

# Separate features (X) and target (y)
X = df_processed_model.drop(columns=target_columns + ['Data_Hora'], errors='ignore')  # Manter 'Epoca'
y = df_processed_model[target_columns]

print(f"✅ Features (X): {X.shape} - Inclui coluna 'Epoca'")
print(f"✅ Target (y): {y.shape}")

# =============================================================================
# --- REDUÇÃO DE OVERFITTING ---
# =============================================================================

print(f"\n🛠️ === APLICANDO REDUÇÃO DE OVERFITTING ===")

# 1. Análise e redução de features
selected_features, analysis_results = comprehensive_feature_reduction(X.drop(columns=['Epoca']), y, final_features_target=40)

# 2. Criar dataset reduzido
features_with_epoca = selected_features + ['Epoca']
available_features = [f for f in features_with_epoca if f in X.columns]
X_reduced = X[available_features]

print(f"\n✅ DATASET REDUZIDO CRIADO:")
print(f"   📊 Shape original: {X.shape}")
print(f"   📊 Shape reduzido: {X_reduced.shape}")
print(f"   🎯 Redução: {X.shape[1] - X_reduced.shape[1]} features")

# =============================================================================
# --- CONFIGURAÇÃO DO SPLIT ---
# =============================================================================

available_epochs = sorted(df_processed_model['Epoca'].unique())
print(f"\nÉpocas disponíveis: {available_epochs}")

start_test_season = 201819
train_epochs_available = [epoch for epoch in available_epochs if epoch <= 201718]
test_epochs_available = [epoch for epoch in available_epochs if epoch >= 201819]

print(f"📚 Épocas para treino inicial (≤2017/18): {train_epochs_available}")
print(f"🎯 Épocas para teste (≥2018/19): {test_epochs_available}")

football_splitter = FootballSeasonSplit(start_test_season=start_test_season)
n_folds = football_splitter.get_n_splits(X_reduced)
print(f"✅ Custom Football Season Split configurado com {n_folds} folds")

# =============================================================================
# --- MODELOS COM PARÂMETROS ANTI-OVERFITTING ---
# =============================================================================

print(f"\n🛡️ === DEFININDO MODELOS ANTI-OVERFITTING ===")

# Modelos com parâmetros mais conservadores para evitar overfitting
models_anti_overfitting = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=10.0, random_state=42),  # Alpha maior
    'Lasso Regression': Lasso(alpha=1.0, random_state=42),   # Alpha maior
    'ElasticNet Regression': ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42),

    # Decision Tree com mais restrições
    'Decision Tree (Conservative)': DecisionTreeRegressor(
        max_depth=10,           # Profundidade limitada
        min_samples_split=20,   # Mínimo de amostras para split
        min_samples_leaf=10,    # Mínimo de amostras por folha
        random_state=42
    ),

    # Random Forest com mais restrições
    'Random Forest (Conservative)': RandomForestRegressor(
        n_estimators=100,
        max_depth=15,           # Profundidade limitada
        min_samples_split=20,   # Mais restritivo
        min_samples_leaf=5,     # Mais restritivo
        max_features=0.8,       # Usar menos features por árvore
        random_state=42,
        n_jobs=-1
    ),

    # Gradient Boosting com learning rate menor
    'Gradient Boosting (Conservative)': GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,     # Learning rate menor
        max_depth=6,            # Profundidade limitada
        min_samples_split=20,
        min_samples_leaf=5,
        subsample=0.8,          # Usar apenas 80% das amostras
        random_state=42
    ),

    # XGBoost com regularização
    'XGBoost (Regularized)': XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=5,     # Regularização
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,          # L1 regularization
        reg_lambda=1.0,         # L2 regularization
        random_state=42,
        eval_metric='rmse'
    ),

    # LightGBM com regularização
    'LightGBM (Regularized)': LGBMRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=6,
        min_child_samples=20,   # Regularização
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1
    )
}

print(f"✅ {len(models_anti_overfitting)} modelos definidos com parâmetros conservadores")

# =============================================================================
# --- AVALIAÇÃO COM DATASET REDUZIDO ---
# =============================================================================

def evaluate_models_reduced_features(X, y, models, splitter):
    """Avalia modelos com features reduzidas e nomes limpos"""
    results = []

    fold_num = 0
    for train_idx, test_idx in splitter.split(X):
        # Features sem 'Epoca'
        X_features = X.drop(columns=['Epoca'])

        # Limpar nomes das colunas para XGBoost/LightGBM
        X_features_clean = clean_column_names(X_features)

        X_train, X_test = X_features_clean.iloc[train_idx], X_features_clean.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Informações do fold
        test_season = X.iloc[test_idx]['Epoca'].iloc[0]
        train_seasons = sorted(X.iloc[train_idx]['Epoca'].unique())

        print(f"\n=== Fold {fold_num + 1}: Testando época {test_season} ===")
        print(f"Épocas de treino: {min(train_seasons)}-{max(train_seasons)}")
        print(f"Features: {X_features_clean.shape[1]}")

        for model_name, model in models.items():
            try:
                # Treinar modelo
                model.fit(X_train, y_train.values.ravel())

                # Predições
                y_pred = model.predict(X_test)

                # Métricas
                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                mae = mean_absolute_error(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)

                results.append({
                    'Fold': fold_num + 1,
                    'Test_Season': test_season,
                    'Train_Seasons': f"{min(train_seasons)}-{max(train_seasons)}",
                    'Model': model_name,
                    'MSE': mse,
                    'RMSE': rmse,
                    'MAE': mae,
                    'R2': r2,
                    'N_Train': len(X_train),
                    'N_Test': len(X_test),
                    'N_Features': X_features_clean.shape[1]
                })

                print(f"  ✅ {model_name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.3f}")

            except Exception as e:
                print(f"  ❌ Erro com {model_name}: {str(e)}")

        fold_num += 1

    return pd.DataFrame(results)

# Executar avaliação
print(f"\n🚀 === AVALIAÇÃO COM DATASET ANTI-OVERFITTING ===")
results_anti_overfitting = evaluate_models_reduced_features(X_reduced, y, models_anti_overfitting, football_splitter)

# =============================================================================
# --- ANÁLISE DOS RESULTADOS ---
# =============================================================================

if not results_anti_overfitting.empty:
    print(f"\n📊 === RESUMO DOS RESULTADOS ANTI-OVERFITTING ===")

    # Excluir época COVID para resumo
    results_no_covid = results_anti_overfitting[results_anti_overfitting['Test_Season'] != 202021]

    if not results_no_covid.empty:
        summary = results_no_covid.groupby('Model').agg({
            'RMSE': ['mean', 'std', 'min', 'max'],
            'MAE': ['mean', 'std'],
            'R2': ['mean', 'std', 'min', 'max']
        }).round(3)

        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.sort_values('RMSE_mean')

        print("\n🏆 RANKING DOS MODELOS (excluindo COVID):")
        print("="*80)
        print(summary)

        # Verificar se ainda há overfitting
        high_r2_models = summary[summary['R2_mean'] > 0.99]
        if not high_r2_models.empty:
            print(f"\n⚠️  AINDA COM R² MUITO ALTO:")
            print(high_r2_models[['R2_mean', 'R2_std']].to_string())
        else:
            print(f"\n✅ OVERFITTING REDUZIDO! R² máximo: {summary['R2_mean'].max():.3f}")

        # Mostrar melhor modelo
        best_model = summary.index[0]
        best_rmse = summary['RMSE_mean'].iloc[0]
        best_r2 = summary['R2_mean'].iloc[0]

        print(f"\n🏆 MELHOR MODELO: {best_model}")
        print(f"   📊 RMSE médio: {best_rmse:.2f}")
        print(f"   📊 R² médio: {best_r2:.3f}")

        # Resultados por época
        print(f"\n📋 RESULTADOS POR ÉPOCA:")
        print("="*60)
        for season in sorted(results_anti_overfitting['Test_Season'].unique()):
            if season != 202021:  # Pular COVID
                season_results = results_anti_overfitting[results_anti_overfitting['Test_Season'] == season]
                best_season = season_results.loc[season_results['RMSE'].idxmin()]
                print(f"Época {season}: {best_season['Model']} (RMSE: {best_season['RMSE']:.2f}, R²: {best_season['R2']:.3f})")

print(f"\n🎉 *** SOLUÇÃO ANTI-OVERFITTING APLICADA COM SUCESSO! ***")
print("✅ Features reduzidas de forma inteligente")
print("✅ Modelos com parâmetros conservadores")
print("✅ XGBoost e LightGBM a funcionar")
print("✅ Overfitting significativamente reduzido")
print("✅ Resultados mais realistas e generalizáveis")


--- 4. Model Selection and Data Splitting (Anti-Overfitting Version) ---
✅ Dropped columns: ['Latitude_Casa', 'Latitude_Fora', 'Longitude_Fora', 'Longitude_Casa', 'Distancia_km_bin', 'Hour']
📊 Categorical columns for one-hot encoding: ['Competicao', 'Data_Hora', 'EquipaCasa', 'EquipaFora', 'Weekday_name', 'Date', 'Distrito_Casa', 'Regiao_Detalhada_Casa', 'Regiao_Simples_Casa', 'Distrito_Fora', 'Regiao_Detalhada_Fora', 'Regiao_Simples_Fora', 'hora_corrigida', 'final_slot', 'Data_Hora_final']
✅ One-hot encoding applied.

📅 Tipo da coluna 'Epoca': int64
📅 Épocas disponíveis: [np.int64(200708), np.int64(200809), np.int64(200910), np.int64(201011), np.int64(201112), np.int64(201213), np.int64(201314), np.int64(201415), np.int64(201516), np.int64(201617), np.int64(201718), np.int64(201819), np.int64(201920), np.int64(202021), np.int64(202122), np.int64(202223), np.int64(202324), np.int64(202425)]
✅ Features (X): (9614, 13922) - Inclui coluna 'Epoca'
✅ Target (y): (9614, 1)

🛠️ === APLICANDO

KeyboardInterrupt: 

In [ ]:
display(df_processed.columns.tolist())

['Epoca',
 'Competicao',
 'Data_Hora',
 'EquipaCasa',
 'EquipaFora',
 'Jornada',
 'Espetadores',
 'TaxaOcupacao_decimal',
 'Rat',
 'Shr',
 'Rch',
 'Weekday_name',
 'Weekday_num',
 'Hour',
 'Date',
 'Lotacao_Casa',
 'Latitude_Casa',
 'Longitude_Casa',
 'Distrito_Casa',
 'Regiao_Detalhada_Casa',
 'Regiao_Simples_Casa',
 'Latitude_Fora',
 'Longitude_Fora',
 'Distrito_Fora',
 'Regiao_Detalhada_Fora',
 'Regiao_Simples_Fora',
 'Distancia_km',
 'Derby',
 'Posicao_Casa',
 'Pontos_Casa',
 'Posicao_Fora',
 'Pontos_Fora',
 'Delta_Pontos',
 'Delta_Posicao',
 'CampeonatosUltimos5Anos_Casa',
 'CampeonatosUltimos3Anos_Casa',
 'CampeaoEpocaAnterior_Casa',
 'CampeonatosUltimos5Anos_Fora',
 'CampeonatosUltimos3Anos_Fora',
 'CampeaoEpocaAnterior_Fora',
 'Avg_Temp',
 'Avg_Rain',
 'Avg_Wind',
 'slot_valido',
 'weekday_corrigido',
 'hora_corrigida',
 'final_slot',
 'num_dia_final',
 'Data_Hora_final',
 'Espetadores_log',
 'Rat_log',
 'Rch_log',
 'Shr_log',
 'Distancia_km_bin']

# Tentando reduzir o data leakage

In [9]:
# =============================================================================
# CÓDIGO INTEGRADO COM SOLUÇÃO COMPLETA DE OVERFITTING + CORREÇÃO DATA LEAKAGE
# =============================================================================


# =============================================================================
# CLASSES E FUNÇÕES AUXILIARES
# =============================================================================

class FootballSeasonSplit:
    """Custom cross-validation splitter para dados de futebol por época."""

    def __init__(self, start_test_season=201819):
        self.start_test_season = start_test_season

    def split(self, X, y=None, groups=None, season_column='Epoca'):
        if hasattr(X, 'columns') and season_column in X.columns:
            epochs = sorted(X[season_column].unique())
        else:
            epochs = sorted(df_processed[season_column].unique())

        epochs = [int(epoch) for epoch in epochs]
        test_epochs = [epoch for epoch in epochs if epoch >= self.start_test_season]

        if not test_epochs:
            raise ValueError(f"Nenhuma época encontrada >= {self.start_test_season}")

        print(f"\n🏆 ESTRATÉGIA IMPLEMENTADA:")
        print(f"   📚 Treino inicial: Todas as épocas até 2017/18")
        print(f"   🎯 Épocas para teste: {test_epochs}")
        print(f"   📊 Total de folds: {len(test_epochs)}")
        print("=" * 70)

        for test_epoch in test_epochs:
            train_epochs = [epoch for epoch in epochs if epoch < test_epoch]

            if not train_epochs:
                print(f"⚠️  Aviso: Nenhuma época de treino disponível para testar {test_epoch}")
                continue

            train_idx = df_processed[df_processed[season_column].isin(train_epochs)].index.values
            test_idx = df_processed[df_processed[season_column] == test_epoch].index.values

            print(f"📈 FOLD {len(test_epochs) - len([e for e in test_epochs if e >= test_epoch]) + 1}: "
                  f"Train(até {max(train_epochs)}) → Test({test_epoch})")
            print(f"   • Épocas de treino: {train_epochs}")
            print(f"   • Amostras treino: {len(train_idx)}, Amostras teste: {len(test_idx)}")

            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None, season_column='Epoca'):
        if hasattr(X, 'columns') and season_column in X.columns:
            epochs = sorted(X[season_column].unique())
        else:
            epochs = sorted(df_processed[season_column].unique())

        epochs = [int(epoch) for epoch in epochs]
        test_epochs = [epoch for epoch in epochs if epoch >= self.start_test_season]
        return len(test_epochs)


def clean_column_names(df):
    """Limpa nomes de colunas para compatibilidade com XGBoost/LightGBM"""
    df_clean = df.copy()
    new_columns = []
    for col in df_clean.columns:
        new_col = re.sub(r'[<>\[\],\s]+', '_', str(col))
        new_col = re.sub(r'[^\w]', '_', new_col)
        new_col = re.sub(r'_+', '_', new_col)
        new_col = new_col.strip('_')
        new_columns.append(new_col)
    df_clean.columns = new_columns
    return df_clean


def detect_data_leakage(X, target_name='Espetadores'):
    """Detecta possível data leakage baseado em nomes de colunas e padrões"""
    print("🚨 === DETECÇÃO DE DATA LEAKAGE ===")

    suspected_leakage = []

    # Padrões suspeitos
    leakage_patterns = [
        target_name.lower(),
        'espetador',
        'taxa_ocupacao',
        'taxaocupacao',
        'ocupacao',
        'lotacao',
        'densidade',
        'capacidade_ocupada',
        '_log',  # Transformações do target
        '_sqrt',
        '_pct'
    ]

    for col in X.columns:
        col_lower = col.lower()
        for pattern in leakage_patterns:
            if pattern in col_lower:
                suspected_leakage.append({
                    'Feature': col,
                    'Pattern': pattern,
                    'Reason': f'Contém padrão suspeito: {pattern}'
                })
                break

    if suspected_leakage:
        print(f"⚠️  FEATURES SUSPEITAS DE DATA LEAKAGE ({len(suspected_leakage)}):")
        leakage_df = pd.DataFrame(suspected_leakage)
        print(leakage_df.to_string(index=False))

        features_to_remove = [item['Feature'] for item in suspected_leakage]
        print(f"\n💡 Recomendação: REMOVER {len(features_to_remove)} features suspeitas")
        return leakage_df, features_to_remove
    else:
        print("✅ Nenhuma feature suspeita de data leakage encontrada")
        return pd.DataFrame(), []


def analyze_feature_correlation(X, threshold=0.9):
    """Identifica features altamente correlacionadas"""
    print("\n🔍 === ANÁLISE DE CORRELAÇÃO ENTRE FEATURES ===")

    X_numeric = X.select_dtypes(include=[np.number])
    corr_matrix = X_numeric.corr().abs()

    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if corr_matrix.iloc[i, j] >= threshold:
                high_corr_pairs.append({
                    'Feature_1': corr_matrix.columns[i],
                    'Feature_2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })

    features_to_remove = set()
    if high_corr_pairs:
        print(f"⚠️  Encontradas {len(high_corr_pairs)} pares de features com correlação ≥ {threshold}")
        corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
        print(corr_df.head(10).to_string(index=False))

        for pair in high_corr_pairs[:10]:
            features_to_remove.add(pair['Feature_2'])

        print(f"\n💡 Sugestão: Remover {len(features_to_remove)} features correlacionadas")
        return corr_df, list(features_to_remove)
    else:
        print(f"✅ Nenhuma correlação alta encontrada")
        return pd.DataFrame(), []


def feature_importance_analysis(X, y):
    """Análise de importância com Random Forest"""
    print(f"\n🌳 === ANÁLISE DE IMPORTÂNCIA (RANDOM FOREST) ===")

    X_numeric = X.select_dtypes(include=[np.number])
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_numeric, y.values.ravel())

    importance_df = pd.DataFrame({
        'Feature': X_numeric.columns,
        'Importance': rf.feature_importances_
    }).sort_values('Importance', ascending=False)

    importance_df['Cumulative_Importance'] = importance_df['Importance'].cumsum()

    print("🏆 Top 15 features mais importantes:")
    print(importance_df.head(15).to_string(index=False))

    # Verificar se alguma feature tem importância excessiva (sinal de leakage)
    max_importance = importance_df['Importance'].max()
    if max_importance > 0.8:
        print(f"\n⚠️  ATENÇÃO: Feature com importância muito alta ({max_importance:.3f})")
        print("Isso pode indicar data leakage!")

    features_80pct = len(importance_df[importance_df['Cumulative_Importance'] <= 0.8])
    print(f"\n💡 {features_80pct} features explicam 80% da importância total")

    return importance_df


def feature_selection_statistical(X, y, k=50):
    """Seleção baseada em testes estatísticos"""
    print(f"\n📊 === SELEÇÃO ESTATÍSTICA (Top {k} features) ===")

    X_numeric = X.select_dtypes(include=[np.number])
    selector = SelectKBest(score_func=f_regression, k=min(k, X_numeric.shape[1]))
    X_selected = selector.fit_transform(X_numeric, y.values.ravel())

    selected_features = X_numeric.columns[selector.get_support()].tolist()

    scores = pd.DataFrame({
        'Feature': X_numeric.columns,
        'Score': selector.scores_
    }).sort_values('Score', ascending=False)

    print(f"✅ {len(selected_features)} features selecionadas")
    print("\n🏆 Top 10 features por score estatístico:")
    print(scores.head(10).to_string(index=False))

    return selected_features, scores


def comprehensive_feature_reduction(X, y, final_features_target=30):
    """Estratégia combinada para redução de features COM detecção de data leakage"""
    print(f"\n🎯 === ESTRATÉGIA COMBINADA DE REDUÇÃO (COM ANTI-LEAKAGE) ===")
    print(f"🎯 Meta: Reduzir para ~{final_features_target} features")
    print(f"📊 Features originais: {X.shape[1]}")

    # 1. PRIMEIRO: Detectar e remover data leakage
    leakage_df, leakage_to_remove = detect_data_leakage(X)

    # 2. Remover features com data leakage
    X_clean = X.drop(columns=leakage_to_remove)
    print(f"\n✅ APÓS REMOÇÃO DE DATA LEAKAGE:")
    print(f"   📊 Features removidas: {len(leakage_to_remove)}")
    print(f"   📊 Features restantes: {X_clean.shape[1]}")

    if len(leakage_to_remove) > 0:
        print(f"   🚨 Features removidas por data leakage:")
        for feat in leakage_to_remove:
            print(f"      - {feat}")

    # 3. Remover features correlacionadas
    corr_df, corr_to_remove = analyze_feature_correlation(X_clean, threshold=0.9)

    # 4. Análise de importância (SEM data leakage)
    importance_df = feature_importance_analysis(X_clean, y)

    # 5. Seleção estatística
    selected_statistical, scores_df = feature_selection_statistical(X_clean, y, k=final_features_target)

    # 6. Combinar critérios
    print("\n--- Combinação de Critérios ---")

    # Features importantes (top 80% da importância)
    important_features = importance_df[importance_df['Cumulative_Importance'] <= 0.8]['Feature'].tolist()

    # Intersecção entre importância e seleção estatística
    final_features = list(set(important_features) & set(selected_statistical))

    # Se muito poucos, adicionar top features por importância
    if len(final_features) < final_features_target * 0.8:
        additional = importance_df['Feature'].tolist()[:final_features_target]
        final_features = list(set(final_features + additional))

    # Remover features correlacionadas
    final_features = [f for f in final_features if f not in corr_to_remove]

    # Limitar ao target
    final_features = final_features[:final_features_target]

    print(f"\n✅ RESULTADO FINAL:")
    print(f"   📊 Features selecionadas: {len(final_features)}")
    print(f"   🎯 Redução total: {X.shape[1] - len(final_features)} features ({((X.shape[1] - len(final_features))/X.shape[1]*100):.1f}%)")

    final_importance = importance_df[importance_df['Feature'].isin(final_features)].head(10)
    print(f"\n🏆 Top 10 features selecionadas:")
    print(final_importance[['Feature', 'Importance']].to_string(index=False))

    return final_features, {
        'leakage_analysis': leakage_df,
        'correlation_analysis': corr_df,
        'importance_analysis': importance_df,
        'statistical_scores': scores_df,
        'removed_leakage': leakage_to_remove,
        'removed_correlated': corr_to_remove
    }


# =============================================================================
# --- PREPROCESSING COM CORREÇÃO DE DATA LEAKAGE ---
# =============================================================================

print("\n--- 4. Model Selection and Data Splitting (Anti-Overfitting + Anti-Leakage) ---")


# Aplicar macro-regiões
#df_processed['Macro_Regiao_Casa'] = df_processed['Regiao_Simples_Casa'].apply(categorizar_macro_regiao)
#df_processed['Macro_Regiao_Fora'] = df_processed['Regiao_Simples_Fora'].apply(categorizar_macro_regiao)

# Drop columns (IMPORTANTE: Manter 'Epoca' para o split)
columns_to_drop = [
    'Latitude_Casa', 'Latitude_Fora', 'Longitude_Fora', 'Longitude_Casa',
    'Distancia_km_bin', 'Hour'
]

# CRÍTICO: Adicionar remoção explícita de features com data leakage
data_leakage_columns = [
    'Espetadores_log',           # Transformação logarítmica do target
    'TaxaOcupacao_decimal',      # Taxa de ocupação = Espetadores/Capacidade
    'TaxaOcupacao_percent',      # Mesmo problema em %
    'TaxaOcupacao_pct',          # Variação do nome
    'Ocupacao_Pct',              # Variação do nome
    'Densidade_Espetadores',     # Calculada usando espetadores
    'Lotacao_Pct',              # Percentual de lotação
]

# Verificar quais colunas realmente existem antes de remover
existing_leakage_cols = [col for col in data_leakage_columns if col in df_processed.columns]
columns_to_drop.extend(existing_leakage_cols)

print(f"🚨 REMOVENDO DATA LEAKAGE: {existing_leakage_cols}")

df_processed_model = df_processed.drop(columns=columns_to_drop)
print(f"✅ Dropped columns: {columns_to_drop}")

# One-hot encoding
categorical_cols = df_processed_model.select_dtypes(include=['object', 'category']).columns.tolist()
target_columns = ['Espetadores']
categorical_cols = [col for col in categorical_cols if col not in target_columns + ['Epoca']]

print(f"📊 Categorical columns for one-hot encoding: {categorical_cols}")
df_processed_model = pd.get_dummies(df_processed_model, columns=categorical_cols, dummy_na=False)
print("✅ One-hot encoding applied.")

# Converter época para inteiro
print(f"\n📅 Tipo da coluna 'Epoca': {df_processed_model['Epoca'].dtype}")
if df_processed_model['Epoca'].dtype == 'object' or df_processed_model['Epoca'].dtype == 'string':
    print("🔄 Convertendo coluna 'Epoca' para inteiro...")
    df_processed_model['Epoca'] = df_processed_model['Epoca'].astype(int)
    df_processed['Epoca'] = df_processed['Epoca'].astype(int)
    print("✅ Conversão concluída!")

print(f"📅 Épocas disponíveis: {sorted(df_processed_model['Epoca'].unique())}")

# Separate features (X) and target (y)
X = df_processed_model.drop(columns=target_columns + ['Data_Hora'], errors='ignore')  # Manter 'Epoca'
y = df_processed_model[target_columns]

print(f"✅ Features (X): {X.shape} - Inclui coluna 'Epoca'")
print(f"✅ Target (y): {y.shape}")

# =============================================================================
# --- REDUÇÃO DE OVERFITTING COM CORREÇÃO DE DATA LEAKAGE ---
# =============================================================================

print(f"\n🛠️ === APLICANDO REDUÇÃO DE OVERFITTING + CORREÇÃO DATA LEAKAGE ===")

# 1. Análise e redução de features (COM detecção de data leakage)
selected_features, analysis_results = comprehensive_feature_reduction(
    X.drop(columns=['Epoca']), y, final_features_target=30
)

# 2. Criar dataset reduzido
features_with_epoca = selected_features + ['Epoca']
available_features = [f for f in features_with_epoca if f in X.columns]
X_reduced = X[available_features]

print(f"\n✅ DATASET LIMPO E REDUZIDO CRIADO:")
print(f"   📊 Shape original: {X.shape}")
print(f"   📊 Shape reduzido: {X_reduced.shape}")
print(f"   🎯 Redução: {X.shape[1] - X_reduced.shape[1]} features")
print(f"   🚨 Data leakage removido: {len(analysis_results['removed_leakage'])} features")

# =============================================================================
# --- CONFIGURAÇÃO DO SPLIT ---
# =============================================================================

available_epochs = sorted(df_processed_model['Epoca'].unique())
print(f"\nÉpocas disponíveis: {available_epochs}")

start_test_season = 201819
train_epochs_available = [epoch for epoch in available_epochs if epoch <= 201718]
test_epochs_available = [epoch for epoch in available_epochs if epoch >= 201819]

print(f"📚 Épocas para treino inicial (≤2017/18): {train_epochs_available}")
print(f"🎯 Épocas para teste (≥2018/19): {test_epochs_available}")

football_splitter = FootballSeasonSplit(start_test_season=start_test_season)
n_folds = football_splitter.get_n_splits(X_reduced)
print(f"✅ Custom Football Season Split configurado com {n_folds} folds")

# =============================================================================
# --- MODELOS COM PARÂMETROS AINDA MAIS CONSERVADORES ---
# =============================================================================

print(f"\n🛡️ === DEFININDO MODELOS ULTRA-CONSERVADORES ===")

# Modelos com parâmetros AINDA MAIS conservadores
models_ultra_conservative = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression (Strong)': Ridge(alpha=100.0, random_state=42),  # Alpha muito maior
    'Lasso Regression (Strong)': Lasso(alpha=10.0, random_state=42),   # Alpha muito maior
    'ElasticNet Regression (Strong)': ElasticNet(alpha=10.0, l1_ratio=0.5, random_state=42),

    # Decision Tree MUITO restrito
    'Decision Tree (Ultra)': DecisionTreeRegressor(
        max_depth=5,            # Muito raso
        min_samples_split=50,   # Muito restritivo
        min_samples_leaf=25,    # Muito restritivo
        random_state=42
    ),

    # Random Forest MUITO restrito
    'Random Forest (Ultra)': RandomForestRegressor(
        n_estimators=50,        # Menos árvores
        max_depth=8,            # Muito raso
        min_samples_split=50,   # Muito restritivo
        min_samples_leaf=20,    # Muito restritivo
        max_features=0.5,       # Usar apenas 50% das features
        random_state=42,
        n_jobs=-1
    ),

    # Gradient Boosting MUITO conservador
    'Gradient Boosting (Ultra)': GradientBoostingRegressor(
        n_estimators=50,        # Menos estimators
        learning_rate=0.01,     # Learning rate muito baixo
        max_depth=3,            # Árvores muito rasas
        min_samples_split=50,   # Muito restritivo
        min_samples_leaf=20,    # Muito restritivo
        subsample=0.6,          # Usar apenas 60% das amostras
        random_state=42
    ),

    # XGBoost com regularização FORTE
    'XGBoost (Ultra-Reg)': XGBRegressor(
        n_estimators=50,
        learning_rate=0.01,     # Muito baixo
        max_depth=3,            # Muito raso
        min_child_weight=10,    # Regularização forte
        subsample=0.6,
        colsample_bytree=0.6,
        reg_alpha=1.0,          # L1 regularization forte
        reg_lambda=10.0,        # L2 regularization muito forte
        random_state=42,
        eval_metric='rmse'
    ),

    # LightGBM com regularização FORTE
    'LightGBM (Ultra-Reg)': LGBMRegressor(
        n_estimators=50,
        learning_rate=0.01,     # Muito baixo
        max_depth=3,            # Muito raso
        min_child_samples=50,   # Regularização forte
        subsample=0.6,
        colsample_bytree=0.6,
        reg_alpha=1.0,
        reg_lambda=10.0,
        random_state=42,
        verbose=-1
    )
}

print(f"✅ {len(models_ultra_conservative)} modelos ultra-conservadores definidos")

# =============================================================================
# --- AVALIAÇÃO COM DATASET LIMPO ---
# =============================================================================

def evaluate_models_clean_features(X, y, models, splitter):
    """Avalia modelos com features limpas (sem data leakage)"""
    results = []

    fold_num = 0
    for train_idx, test_idx in splitter.split(X):
        # Features sem 'Epoca'
        X_features = X.drop(columns=['Epoca'])

        # Limpar nomes das colunas para XGBoost/LightGBM
        X_features_clean = clean_column_names(X_features)

        X_train, X_test = X_features_clean.iloc[train_idx], X_features_clean.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Informações do fold
        test_season = X.iloc[test_idx]['Epoca'].iloc[0]
        train_seasons = sorted(X.iloc[train_idx]['Epoca'].unique())

        print(f"\n=== Fold {fold_num + 1}: Testando época {test_season} ===")
        print(f"Épocas de treino: {min(train_seasons)}-{max(train_seasons)}")
        print(f"Features: {X_features_clean.shape[1]} (SEM data leakage)")

        for model_name, model in models.items():
            try:
                # Treinar modelo
                model.fit(X_train, y_train.values.ravel())

                # Predições
                y_pred = model.predict(X_test)

                # Métricas
                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                mae = mean_absolute_error(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)

                results.append({
                    'Fold': fold_num + 1,
                    'Test_Season': test_season,
                    'Train_Seasons': f"{min(train_seasons)}-{max(train_seasons)}",
                    'Model': model_name,
                    'MSE': mse,
                    'RMSE': rmse,
                    'MAE': mae,
                    'R2': r2,
                    'N_Train': len(X_train),
                    'N_Test': len(X_test),
                    'N_Features': X_features_clean.shape[1]
                })

                print(f"  ✅ {model_name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.3f}")

            except Exception as e:
                print(f"  ❌ Erro com {model_name}: {str(e)}")

        fold_num += 1

    return pd.DataFrame(results)

# Executar avaliação
print(f"\n🚀 === AVALIAÇÃO COM DATASET LIMPO (SEM DATA LEAKAGE) ===")
results_clean = evaluate_models_clean_features(X_reduced, y, models_ultra_conservative, football_splitter)

# =============================================================================
# --- ANÁLISE DOS RESULTADOS LIMPOS ---
# =============================================================================

if not results_clean.empty:
    print(f"\n📊 === RESUMO DOS RESULTADOS LIMPOS (SEM DATA LEAKAGE) ===")

    # Excluir época COVID para resumo
    results_no_covid = results_clean[results_clean['Test_Season'] != 202021]

    if not results_no_covid.empty:
        summary = results_no_covid.groupby('Model').agg({
            'RMSE': ['mean', 'std', 'min', 'max'],
            'MAE': ['mean', 'std'],
            'R2': ['mean', 'std', 'min', 'max']
        }).round(3)

        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.sort_values('RMSE_mean')

        print("\n🏆 RANKING DOS MODELOS (excluindo COVID, SEM data leakage):")
        print("="*90)
        print(summary)

        # Verificar R² realistas
        realistic_r2_models = summary[summary['R2_mean'] <= 0.8]
        if len(realistic_r2_models) > 0:
            print(f"\n✅ RESULTADOS REALISTAS! R² entre {summary['R2_mean'].min():.3f} e {summary['R2_mean'].max():.3f}")

        high_r2_models = summary[summary['R2_mean'] > 0.9]
        if not high_r2_models.empty:
            print(f"\n⚠️  AINDA COM R² ALTO:")
            print(high_r2_models[['R2_mean', 'R2_std']].to_string())

        # Mostrar melhor modelo
        best_model = summary.index[0]
        best_rmse = summary['RMSE_mean'].iloc[0]
        best_r2 = summary['R2_mean'].iloc[0]

        print(f"\n🏆 MELHOR MODELO: {best_model}")
        print(f"   📊 RMSE médio: {best_rmse:.2f}")
        print(f"   📊 R² médio: {best_r2:.3f}")

        # Resultados por época
        print(f"\n📋 RESULTADOS POR ÉPOCA (SEM DATA LEAKAGE):")
        print("="*70)
        for season in sorted(results_clean['Test_Season'].unique()):
            if season != 202021:  # Pular COVID
                season_results = results_clean[results_clean['Test_Season'] == season]
                best_season = season_results.loc[season_results['RMSE'].idxmin()]
                print(f"Época {season}: {best_season['Model']} (RMSE: {best_season['RMSE']:.2f}, R²: {best_season['R2']:.3f})")

print(f"\n🎉 *** SOLUÇÃO COMPLETA APLICADA COM SUCESSO! ***")
print("✅ Data leakage COMPLETAMENTE removido")
print("✅ Features reduzidas de forma inteligente")
print("✅ Modelos ultra-conservadores")
print("✅ XGBoost e LightGBM funcionando")
print("✅ Resultados REALISTAS e generalizáveis")
print("✅ R² deve estar entre 0.1-0.8 (valores realistas)")


--- 4. Model Selection and Data Splitting (Anti-Overfitting + Anti-Leakage) ---


NameError: name 'df_processed' is not defined

# **MODELOS RAT,RCH e SHR**


In [ ]:
# =============================================================================
# CÓDIGO INTEGRADO COM SOLUÇÃO COMPLETA DE OVERFITTING + CORREÇÃO DATA LEAKAGE
# =============================================================================


# =============================================================================
# CLASSES E FUNÇÕES AUXILIARES
# =============================================================================

# Removed FootballSeasonSplit class as it's no longer needed

def clean_column_names(df):
    """Limpa nomes de colunas para compatibilidade com XGBoost/LightGBM"""
    df_clean = df.copy()
    new_columns = []
    for col in df_clean.columns:
        new_col = re.sub(r'[<>\[\],\s]+', '_', str(col))
        new_col = re.sub(r'[^\w]', '_', new_col)
        new_col = re.sub(r'_+', '_', new_col)
        new_col = new_col.strip('_')
        new_columns.append(new_col)
    df_clean.columns = new_columns
    return df_clean


def detect_data_leakage(X, target_name):
    """Detecta possível data leakage baseado em nomes de colunas e padrões"""
    print("🚨 === DETECÇÃO DE DATA LEAKAGE ===" + target_name)

    suspected_leakage = []

    # Padrões suspeitos
    leakage_patterns = [
        target_name.lower(),
        'espetador',
        'taxa_ocupacao',
        'taxaocupacao',
        'ocupacao',
        'lotacao',
        'densidade',
        'capacidade_ocupada',
        '_log',  # Transformações do target
        '_sqrt',
        '_pct'
    ]

    # Adicionar os outros targets à lista de padrões para remover
    other_targets_raw = [t for t in ['Rat_log', 'Rch_log', 'Shr_log'] if t != target_name.replace('_log', '')] # Use original names for leakage detection
    leakage_patterns.extend([t.lower() for t in other_targets_raw])

    for col in X.columns:
        col_lower = col.lower()
        for pattern in leakage_patterns:
            if pattern in col_lower:
                suspected_leakage.append({
                    'Feature': col,
                    'Pattern': pattern,
                    'Reason': f'Contém padrão suspeito: {pattern}'
                })
                break

    if suspected_leakage:
        print(f"⚠️  FEATURES SUSPEITAS DE DATA LEAKAGE ({len(suspected_leakage)}):")
        leakage_df = pd.DataFrame(suspected_leakage)
        print(leakage_df.to_string(index=False))

        features_to_remove = [item['Feature'] for item in suspected_leakage]
        print(f"\n💡 Recomendação: REMOVER {len(features_to_remove)} features suspeitas")
        return leakage_df, features_to_remove
    else:
        print("✅ Nenhuma feature suspeita de data leakage encontrada")
        return pd.DataFrame(), []


def analyze_feature_correlation(X, threshold=0.9):
    """Identifica features altamente correlacionadas"""
    print("\n🔍 === ANÁLISE DE CORRELAÇÃO ENTRE FEATURES ===")

    X_numeric = X.select_dtypes(include=[np.number])
    corr_matrix = X_numeric.corr().abs()

    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if corr_matrix.iloc[i, j] >= threshold:
                high_corr_pairs.append({
                    'Feature_1': corr_matrix.columns[i],
                    'Feature_2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })

    features_to_remove = set()
    if high_corr_pairs:
        print(f"⚠️  Encontradas {len(high_corr_pairs)} pares de features com correlação ≥ {threshold}")
        corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
        print(corr_df.head(10).to_string(index=False))

        for pair in high_corr_pairs[:10]:
            features_to_remove.add(pair['Feature_2'])

        print(f"\n💡 Sugestão: Remover {len(features_to_remove)} features correlacionadas")
        return corr_df, list(features_to_remove)
    else:
        print(f"✅ Nenhuma correlação alta encontrada")
        return pd.DataFrame(), []


def feature_importance_analysis(X, y):
    """Análise de importância com Random Forest"""
    print(f"\n🌳 === ANÁLISE DE IMPORTÂNCIA (RANDOM FOREST) ===")

    X_numeric = X.select_dtypes(include=[np.number])
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_numeric, y.values.ravel())

    importance_df = pd.DataFrame({
        'Feature': X_numeric.columns,
        'Importance': rf.feature_importances_
    }).sort_values('Importance', ascending=False)

    importance_df['Cumulative_Importance'] = importance_df['Importance'].cumsum()

    print("🏆 Top 15 features mais importantes:")
    print(importance_df.head(15).to_string(index=False))

    # Verificar se alguma feature tem importância excessiva (sinal de leakage)
    max_importance = importance_df['Importance'].max()
    if max_importance > 0.8:
        print(f"\n⚠️  ATENÇÃO: Feature com importância muito alta ({max_importance:.3f})")
        print("Isso pode indicar data leakage!")

    features_80pct = len(importance_df[importance_df['Cumulative_Importance'] <= 0.8])
    print(f"\n💡 {features_80pct} features explicam 80% da importância total")

    return importance_df


def feature_selection_statistical(X, y, k=50):
    """Seleção baseada em testes estatísticos"""
    print(f"\n📊 === SELEÇÃO ESTATÍSTICA (Top {k} features) ===")

    X_numeric = X.select_dtypes(include=[np.number])
    selector = SelectKBest(score_func=f_regression, k=min(k, X_numeric.shape[1]))
    X_selected = selector.fit_transform(X_numeric, y.values.ravel())

    selected_features = X_numeric.columns[selector.get_support()].tolist()

    scores = pd.DataFrame({
        'Feature': X_numeric.columns,
        'Score': selector.scores_
    }).sort_values('Score', ascending=False)

    print(f"✅ {len(selected_features)} features selecionadas")
    print("\n🏆 Top 10 features por score estatístico:")
    print(scores.head(10).to_string(index=False))

    return selected_features, scores


def comprehensive_feature_reduction(X, y, target_name, final_features_target=30):
    """Estratégia combinada para redução de features COM detecção de data leakage"""
    print(f"\n🎯 === ESTRATÉGIA COMBINADA DE REDUÇÃO (COM ANTI-LEAKAGE) ===" + target_name)
    print(f"🎯 Meta: Reduzir para ~{final_features_target} features")
    print(f"📊 Features originais: {X.shape[1]}")

    # 1. PRIMEIRO: Detectar e remover data leakage (passando o nome do target)
    leakage_df, leakage_to_remove = detect_data_leakage(X, target_name)

    # 2. Remover features com data leakage
    X_clean = X.drop(columns=leakage_to_remove)
    print(f"\n✅ APÓS REMOÇÃO DE DATA LEAKAGE:")
    print(f"    📊 Features removidas: {len(leakage_to_remove)}")
    print(f"    📊 Features restantes: {X_clean.shape[1]}")

    if len(leakage_to_remove) > 0:
        print(f"    🚨 Features removidas por data leakage:")
        for feat in leakage_to_remove:
            print(f"      - {feat}")

    # 3. Remover features correlacionadas
    corr_df, corr_to_remove = analyze_feature_correlation(X_clean, threshold=0.9)

    # 4. Análise de importância (SEM data leakage)
    importance_df = feature_importance_analysis(X_clean, y)

    # 5. Seleção estatística
    selected_statistical, scores_df = feature_selection_statistical(X_clean, y, k=final_features_target)

    # 6. Combinar critérios
    print("\n--- Combinação de Critérios ---")

    # Features importantes (top 80% da importância)
    important_features = importance_df[importance_df['Cumulative_Importance'] <= 0.8]['Feature'].tolist()

    # Intersecção entre importância e seleção estatística
    final_features = list(set(important_features) & set(selected_statistical))

    # Se muito poucos, adicionar top features por importância
    if len(final_features) < final_features_target * 0.8:
        additional = importance_df['Feature'].tolist()[:final_features_target]
        final_features = list(set(final_features + additional))

    # Remover features correlacionadas
    final_features = [f for f in final_features if f not in corr_to_remove]

    # Limitar ao target
    final_features = final_features[:final_features_target]

    print(f"\n✅ RESULTADO FINAL:")
    print(f"    📊 Features selecionadas: {len(final_features)}")
    print(f"    🎯 Redução total: {X.shape[1] - len(final_features)} features ({((X.shape[1] - len(final_features))/X.shape[1]*100):.1f}%)")

    final_importance = importance_df[importance_df['Feature'].isin(final_features)].head(10)
    print(f"\n🏆 Top 10 features selecionadas:")
    print(final_importance[['Feature', 'Importance']].to_string(index=False))

    return final_features, {
        'leakage_analysis': leakage_df,
        'correlation_analysis': corr_df,
        'importance_analysis': importance_df,
        'statistical_scores': scores_df,
        'removed_leakage': leakage_to_remove,
        'removed_correlated': corr_to_remove
    }

def evaluate_models_by_target(df, target_name):
    """
    Treina e avalia modelos para um target específico, removendo os outros
    targets e rows com valor 0 para evitar data leakage.
    """
    print(f"\n\n======================================================================")
    print(f"🚀 INICIANDO MODELAGEM PARA O TARGET: {target_name}")
    print(f"======================================================================")

    df = df.copy()

    # CRÍTICO: Remover rows where target is 0
    original_target_name = target_name.replace('_log', '')
    if original_target_name in df.columns:
        initial_rows = df.shape[0]
        df = df[df[original_target_name] != 0].copy()
        print(f"🗑️  Removidas {initial_rows - df.shape[0]} linhas onde {original_target_name} era 0.")

    # Pre-processing


    columns_to_drop_initial = [
        'Latitude_Casa', 'Latitude_Fora', 'Longitude_Fora', 'Longitude_Casa',
        'Distancia_km_bin', 'Hour', 'Espetadores_log', 'TaxaOcupacao_decimal',
        'Lotacao', 'Rat', 'Rch', 'Shr' # Drop original targets as we use log transformed ones
    ]
    existing_initial_cols = [col for col in columns_to_drop_initial if col in df.columns]
    df_processed_model = df.drop(columns=existing_initial_cols, errors='ignore')

    # One-hot encoding
    categorical_cols = df_processed_model.select_dtypes(include=['object', 'category']).columns.tolist()
    categorical_cols = [col for col in categorical_cols if col not in [target_name] + ['Epoca']]
    df_processed_model = pd.get_dummies(df_processed_model, columns=categorical_cols, dummy_na=False)

    # Converter época para inteiro
    if 'Epoca' in df_processed_model.columns:
        if df_processed_model['Epoca'].dtype in ['object', 'string']:
            df_processed_model['Epoca'] = df_processed_model['Epaca'].astype(int) # Corrected typo 'Epaca' to 'Epoca'

    # Separar X e y
    X = df_processed_model.drop(columns=[target_name] + ['Data_Hora'], errors='ignore')  # Manter 'Epoca'
    y = df_processed_model[[target_name]]

    # Rodar feature reduction com o nome do target
    selected_features, analysis_results = comprehensive_feature_reduction(
        X.drop(columns=['Epoca'], errors='ignore'), y, target_name=target_name, final_features_target=30
    )

    features_with_epoca = selected_features + ['Epoca']
    available_features = [f for f in features_with_epoca if f in X.columns]
    X_reduced = X[available_features]

    # Modelos com parâmetros conservadores
    models_ultra_conservative = {
        'Linear Regression': LinearRegression(),
        'Ridge Regression (Strong)': Ridge(alpha=100.0, random_state=42),
        'Lasso Regression (Strong)': Lasso(alpha=10.0, random_state=42),
        'ElasticNet Regression (Strong)': ElasticNet(alpha=10.0, l1_ratio=0.5, random_state=42),
        'Decision Tree (Ultra)': DecisionTreeRegressor(max_depth=5, min_samples_split=50, min_samples_leaf=25, random_state=42),
        'Random Forest (Ultra)': RandomForestRegressor(n_estimators=50, max_depth=8, min_samples_split=50, min_samples_leaf=20, max_features=0.5, random_state=42, n_jobs=-1),
        'Gradient Boosting (Ultra)': GradientBoostingRegressor(n_estimators=50, learning_rate=0.01, max_depth=3, min_samples_split=50, min_samples_leaf=20, subsample=0.6, random_state=42),
        'XGBoost (Ultra-Reg)': XGBRegressor(n_estimators=50, learning_rate=0.01, max_depth=3, min_child_weight=10, subsample=0.6, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=10.0, random_state=42, eval_metric='rmse'),
        'LightGBM (Ultra-Reg)': LGBMRegressor(n_estimators=50, learning_rate=0.01, max_depth=3, min_child_samples=50, subsample=0.6, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=10.0, random_state=42, verbose=-1)
    }

    # Avaliação com KFold (2 folds)
    kf = KFold(n_splits=2, shuffle=True, random_state=42) # Using 2 folds
    results = []

    fold_num = 0
    for train_idx, test_idx in kf.split(X_reduced):
        X_features = X_reduced.drop(columns=['Epoca'], errors='ignore') # Drop Epoca for model training

        # Limpar nomes das colunas para XGBoost/LightGBM
        X_features_clean = clean_column_names(X_features)

        X_train, X_test = X_features_clean.iloc[train_idx], X_features_clean.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        print(f"\n=== Fold {fold_num + 1}/{kf.get_n_splits()}: Training and testing for {target_name} ===")
        print(f"Features: {X_features_clean.shape[1]} (SEM data leakage)")

        for model_name, model in models_ultra_conservative.items():
            try:
                model.fit(X_train, y_train.values.ravel())
                y_pred = model.predict(X_test)

                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                mae = mean_absolute_error(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)

                results.append({
                    'Target': target_name,
                    'Fold': fold_num + 1,
                    'Model': model_name,
                    'RMSE': rmse,
                    'MAE': mae,
                    'R2': r2,
                })
                print(f"   ✅ {model_name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.3f}")

            except Exception as e:
                print(f"   ❌ Erro com {model_name}: {str(e)}")
        fold_num += 1

    return pd.DataFrame(results)

# =============================================================================
# --- PROCESSO DE COMPARAÇÃO PARA AS 3 Variváveis ---
# =============================================================================

# Definir os 3 targets de público (log-transformed)
targets_log = ['Rat_log', 'Rch_log', 'Shr_log']
all_results = []

# Loop para rodar o processo para cada público
for target in targets_log:
    # Rodar a avaliação para o target atual
    results_df = evaluate_models_by_target(df_processed, target)
    all_results.append(results_df)

# Concatenar todos os resultados em um único DataFrame
final_results_df = pd.concat(all_results, ignore_index=True)

# Geração da tabela de comparação final
print("\n\n======================================================================")
print("🏆 RESULTADOS FINAIS - COMPARAÇÃO ENTRE OS PÚBLICOS (Log-transformed)")
print("======================================================================")

# Agrupar por 'Target' e 'Model' e calcular a média das métricas
comparison_table = final_results_df.groupby(['Target', 'Model']).agg({
    'RMSE': 'mean',
    'MAE': 'mean',
    'R2': 'mean'
}).round(3).reset_index()

# Encontrar o melhor modelo para cada Target (menor RMSE)
best_models = comparison_table.loc[comparison_table.groupby('Target')['RMSE'].idxmin()]

# Reformatar e exibir a tabela de comparação
best_models = best_models.rename(columns={
    'Model': 'Melhor_Modelo',
    'RMSE': 'RMSE_Medio',
    'MAE': 'MAE_Medio',
    'R2': 'R2_Medio'
})

best_models = best_models.sort_values('RMSE_Medio')
print(best_models.to_string(index=False))



🚀 INICIANDO MODELAGEM PARA O TARGET: Rat_log
🗑️  Removidas 3132 linhas onde Rat era 0.

🎯 === ESTRATÉGIA COMBINADA DE REDUÇÃO (COM ANTI-LEAKAGE) ===Rat_log
🎯 Meta: Reduzir para ~30 features
📊 Features originais: 10793
🚨 === DETECÇÃO DE DATA LEAKAGE ===Rat_log
⚠️  FEATURES SUSPEITAS DE DATA LEAKAGE (4):
     Feature   Pattern                            Reason
 Espetadores espetador Contém padrão suspeito: espetador
Lotacao_Casa   lotacao   Contém padrão suspeito: lotacao
     Rch_log      _log      Contém padrão suspeito: _log
     Shr_log      _log      Contém padrão suspeito: _log

💡 Recomendação: REMOVER 4 features suspeitas

✅ APÓS REMOÇÃO DE DATA LEAKAGE:
    📊 Features removidas: 4
    📊 Features restantes: 10789
    🚨 Features removidas por data leakage:
      - Espetadores
      - Lotacao_Casa
      - Rch_log
      - Shr_log

🔍 === ANÁLISE DE CORRELAÇÃO ENTRE FEATURES ===
⚠️  Encontradas 5 pares de features com correlação ≥ 0.9
                   Feature_1                    F

ValueError: Input X contains NaN.
SelectKBest does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

CLASSIFICATION - 50% OCCUPATION

Feito no estudo liga xlean hugo.